In [1]:
import pandas as pd
import numpy as np
import numba

from numba import jit
from tqdm.auto import tqdm

In [2]:
numba.set_num_threads(16)

In [3]:
dataset_path = "D:/Repo/epi-thesis/dataset/"
working_path = "D:/Repo/epi-thesis/workflow/03. Encoding New Dataset, XGBoost/"

In [4]:
def load_histone_data(file_name, type):
    columns = ['chrom', 'chromStart', 'chromEnd', 'name']
    df = pd.read_csv(file_name, sep="\t", header=None, names=columns)
    df["type"] = type
    return df.to_numpy()

In [5]:
def encode_histone(chrom, tss_start, tss_end, current_histone, threshold = 0.8):
    selected_histone = current_histone[
                    (current_histone[:, 0] == chrom) & 
                    (
                        # Inside the +/- 2KB from TSS
                        ((current_histone[:, 1] >= tss_start) & (current_histone[:, 2] <= tss_end)) |
                        # Overlap at the start
                        ((current_histone[:, 1] < tss_start) & (current_histone[:, 2] > tss_start) & (current_histone[:, 2] < tss_end) & ((current_histone[:, 2] - tss_start)/146 >= threshold)) |
                        # Overlap at the end
                        ((current_histone[:, 1] > tss_start) & (current_histone[:, 1] < tss_end) & (current_histone[:, 2] > tss_end) & ((tss_end - current_histone[:, 1])/146 >= threshold))
                    )
                ]

    arr = np.zeros(4000)
    idx = [x for x in selected_histone[:, 1] - (tss_start)]
    arr[idx] = 1
    return arr

# Load Data

## Load Gene and Its Expression

In [ ]:
gene_df = pd.read_csv(f"{dataset_path}histone_count_overlap80.csv", sep="\t")
display(gene_df.head())
display(gene_df.shape)

In [ ]:
gene_exp_arr = gene_df[['h_chrom', 'h_chromStart', 'h_chromEnd', 'h_value_1', 'n_strand', 'tss', 'tss_start', 'tss_end']].to_numpy()
display(gene_exp_arr)

## Load Histone

In [ ]:
h3k4me3_arr = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K4me3.peak.bed", "h3k4me3")
h3k9ac_arr = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K9ac.peak.bed", "h3k9ac")
h3k9me3_arr = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K9me3.peak.bed", "h3k9me3")
h3k27ac_arr = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K27ac.peak.bed", "h3k27ac")
h3k27me3_arr = load_histone_data(f"{dataset_path}HepG2_Male.histone.H3K27me3.peak.bed", "h3k27me3")

In [ ]:
display(h3k4me3_arr)
display(h3k4me3_arr.shape)

In [ ]:
h3k4me3_stack = np.empty((0, 4000))

In [ ]:
for row in tqdm(gene_exp_arr, "Processing"):
    chrom = row[0]
    tss_start = row[6]
    tss_end = row[7]

    arr = encode_histone(chrom, tss_start, tss_end, h3k4me3_arr)
    h3k4me3_stack = np.vstack((h3k4me3_stack, arr))

In [ ]:
gene_1_10 = gene_exp_arr[0:10]
display(gene_1_10)

In [ ]:
h3k4me3_stack.shape

In [ ]:
array_data = np.array(h3k4me3_stack, dtype=object)

In [ ]:
array_data

In [ ]:
gene_1_10_df = pd.DataFrame(data = gene_1_10, columns=[['h_chrom', 'h_chromStart', 'h_chromEnd', 'h_value_1', 'n_strand', 'tss', 'tss_start', 'tss_end']])

In [ ]:
gene_1_10_df

In [ ]:
print(h3k4me3_stack)

In [ ]:
gene_1_10_df['h3k4me3'] = h3k4me3_stack.tolist()

In [ ]:
gene_1_10_df